# 03 · Qualidade de Dados na Camada Bronze## 1. Objetivo e métodoEsta etapa verifica a qualidade dos dados capturados na Bronze, antes de qualquertransformação. A ordem não é arbitrária: a verificação informa a transformação. Se aSilver fosse construída primeiro, a fonte de cada indicador seria escolhida sem sabero que há de errado nos dados — e a escolha óbvia seria, em vários casos, a errada.### Estratégia: um pilar completo antes de cinco pela metadeO trabalho tem seis métricas alimentadas por sete fontes. Cobrir todas em profundidadesimultaneamente produziria seis análises rasas e nenhum pipeline validado de ponta aponta.A escolha foi outra: **fechar o ciclo completo de um indicador — DEC e FEC — da Bronzeaté a Gold, e usar o resultado como molde para os demais**. Um pilar inteiro exercitatodas as decisões de arquitetura em um caso real: composição de indicador a partir deparcelas, agregação ponderada, mudança de granularidade, construção de dimensão e fato,cálculo de evolução e ranqueamento. As métricas seguintes reaproveitam esse desenho emvez de reinventá-lo.O DEC e o FEC foram escolhidos como primeiro pilar por serem os indicadores maiscomplexos do conjunto: vêm em formato longo, exigem composição normativa a partir deonze parcelas, mudam de regra no meio da série histórica e precisam de agregaçãoponderada para subir de conjunto para distribuidora. O que funcionar aqui funcionapara os outros.### Consequência do prazoA entrega tem data fixa. Fontes que não forem cobertas em profundidade têm a análisede qualidade registrada como pendência explícita, com o que falta verificar e por quê,na seção de fechamento deste notebook. Isso é escolha declarada, não omissão — e aautoavaliação discute o que ficou de fora.### Os cinco critériosA verificação segue os critérios da especificação, aplicados a cada fonte:| Critério | Pergunta ||---|---|| Completude | Existem valores nulos ou vazios? Em que proporção? || Consistência | Os valores seguem o padrão esperado, inclusive o do dicionário de dados? || Unicidade | Existem duplicatas onde não deveria haver? || Acurácia | Os valores fazem sentido no contexto e contra a regra de negócio? || Outliers | Existem valores extremos capazes de distorcer a análise? |Cada fonte ocupa uma seção própria, com as mesmas sete subseções: estrutura, os cincocritérios e uma síntese com os tratamentos definidos. Novas fontes entram como seçõesirmãs, sem alterar a numeração das existentes.

## 2. ConfiguraçãoConstantes da janela de análise e o registro de achados, compartilhado por todas asseções. Cada verificação grava seu resultado em `FINDINGS`, e a síntese de cada fonteconsolida o que foi encontrado.

In [ ]:
import os
import sys

from pyspark.sql import functions as F
from pyspark.sql import Window

REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.config import CATALOG, SCHEMA_BRONZE

BRONZE = f"{CATALOG}.{SCHEMA_BRONZE}"

# Analysis window: full calendar years. Accumulating twelve months always in December
# avoids the distortion caused by mid-series changes in the set of consumer units,
# which distributors usually apply in January.
ANO_INICIAL = 2023
ANO_FINAL = 2025
ANOS_JANELA = list(range(ANO_INICIAL, ANO_FINAL + 1))

# Rounding tolerance accepted when comparing against ANEEL published aggregates
TOLERANCIA = 0.05

# Normative composition of DEC and FEC, PRODIST Module 8, Section 8.2.
# Two regimes: external parcels left the composition from 2022 onwards.
COMPOSICAO = {
    "ate_2021": ["IND", "IP", "XN", "XP"],
    "desde_2022": ["IND", "IP"],
}
ANO_MUDANCA_REGIME = 2022

# Own indicator: internal origin, unplanned, including ISE and Critical Day
PARCELAS_FI = ["IND", "INE", "INC"]

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA_BRONZE}")

print(f"Catalogo........: {BRONZE}")
print(f"Janela..........: {ANO_INICIAL} a {ANO_FINAL} (anos civis)")
print(f"Tolerancia......: {TOLERANCIA}")

In [ ]:
# One row per quality check, consumed by the synthesis of each source section
FINDINGS = []

STATUS_OK = "OK"
STATUS_ATENCAO = "ATENCAO"
STATUS_PROBLEMA = "PROBLEMA"


def check(fonte, criterio, teste, status, detalhe, tratamento=""):
    """Register and print the result of one quality check.

    `fonte` is the Bronze table, `criterio` one of the five specification criteria,
    `teste` a short label, `status` one of the STATUS_* constants, `detalhe` what was
    measured and `tratamento` the decision taken when a problem was found.
    """
    FINDINGS.append({
        "fonte": fonte,
        "criterio": criterio,
        "teste": teste,
        "status": status,
        "detalhe": detalhe,
        "tratamento": tratamento,
    })
    print(f"[{status:<8}] {criterio:<12} {teste}")
    print(f"{'':>11}{detalhe}")
    if tratamento:
        print(f"{'':>11}Tratamento: {tratamento}")
    print()


def resumo_achados(fonte):
    """Print the consolidated findings of one source."""
    linhas = [f for f in FINDINGS if f["fonte"] == fonte]
    if not linhas:
        print("Nenhuma verificacao registrada.")
        return

    por_status = {}
    for f in linhas:
        por_status[f["status"]] = por_status.get(f["status"], 0) + 1

    print(f"{fonte}: {len(linhas)} verificacoes")
    for s in (STATUS_OK, STATUS_ATENCAO, STATUS_PROBLEMA):
        if s in por_status:
            print(f"  {s:<10} {por_status[s]}")
    print()
    return spark.createDataFrame(linhas)

## 3. Panorama das tabelas BronzeVisão rasa de todas as tabelas carregadas: volume, número de colunas e tipos. Servepara dimensionar o conjunto e garantir que nenhuma fonte fique sem menção, mesmoaquelas cuja verificação aprofundada não couber no prazo.

In [ ]:
tabelas = [r["tableName"] for r in spark.sql(f"SHOW TABLES IN {BRONZE}").collect()
           if not r["tableName"].startswith("_")]

panorama = []
for nome in sorted(tabelas):
    df = spark.table(f"{BRONZE}.{nome}")
    tipos = {}
    for _, dtype in df.dtypes:
        tipos[dtype] = tipos.get(dtype, 0) + 1
    panorama.append({
        "tabela": nome,
        "linhas": df.count(),
        "colunas": len(df.columns),
        "tipos": ", ".join(f"{k}:{v}" for k, v in sorted(tipos.items())),
    })

display(spark.createDataFrame(panorama))

## 4. Continuidade — `continuity_indicators`Fonte dos indicadores DEC e FEC, base das métricas 3, 9 e 10. É a primeira a serverificada em profundidade, conforme a estratégia da seção 1.

### 4.1 Estrutura e domíniosA base vem em formato longo: uma linha por conjunto de unidades consumidoras, ano, mêse tipo de indicador. O valor está sempre em `VlrIndiceEnviado`, e o que ele significadepende de `SigIndicador`.Essa estrutura tem uma consequência prática: o mesmo campo carrega grandezas diferentes— horas de interrupção no DEC, número de interrupções no FEC e contagem de clientes no`NumCon`. Qualquer agregação sem filtrar por `SigIndicador` produz número sem sentido.

In [ ]:
FONTE = "continuity_indicators"
cont = spark.table(f"{BRONZE}.{FONTE}")

print("=== schema ===")
for nome, tipo in cont.dtypes:
    print(f"  {nome:<26} {tipo}")

linhas = cont.count()
grao = cont.select("NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice", "NumPeriodoIndice").distinct().count()
print(f"\nlinhas.....................: {linhas:,}")
print(f"grao conjunto x ano x mes..: {grao:,}")
print(f"indicadores por grao.......: {linhas / grao:.1f}")

In [ ]:
# Domain of SigIndicador, with volume and coverage per year
dominio = (cont
    .withColumn("SigIndicador", F.trim("SigIndicador"))
    .groupBy("SigIndicador")
    .agg(F.count("*").alias("linhas"),
         F.min("AnoIndice").alias("primeiro_ano"),
         F.max("AnoIndice").alias("ultimo_ano"),
         F.sum(F.when(F.col("VlrIndiceEnviado") > 0, 1).otherwise(0)).alias("linhas_com_valor"))
    .orderBy("SigIndicador"))

display(dominio)

In [ ]:
# Period coverage: which months exist in each year
cobertura = (cont
    .groupBy("AnoIndice")
    .agg(F.min("NumPeriodoIndice").alias("primeiro_mes"),
         F.max("NumPeriodoIndice").alias("ultimo_mes"),
         F.countDistinct("NumPeriodoIndice").alias("meses"),
         F.countDistinct("IdeConjUndConsumidoras").alias("conjuntos"),
         F.countDistinct("NumCNPJ").alias("distribuidoras"))
    .orderBy("AnoIndice"))

display(cobertura)

anos = [r["AnoIndice"] for r in cobertura.collect()]
incompletos = [r["AnoIndice"] for r in cobertura.collect() if r["meses"] < 12]
check(FONTE, "Estrutura", "cobertura temporal",
      STATUS_OK if all(a in anos for a in ANOS_JANELA) else STATUS_PROBLEMA,
      f"Serie de {min(anos)} a {max(anos)}. Anos com menos de 12 meses: {incompletos or 'nenhum'}. "
      f"A janela {ANO_INICIAL}-{ANO_FINAL} esta integralmente coberta.",
      "Anos parciais ficam fora da janela por construcao, ao adotar anos civis completos.")

### 4.2 CompletudeDuas perguntas. A primeira é a trivial: existem nulos nas colunas? A segunda importamais: cada parcela do DEC e do FEC está presente em todas as combinações de conjunto,ano e mês? Uma parcela ausente não gera nulo — ela simplesmente não vira linha, e acomposição a trataria como zero, subestimando o indicador sem qualquer aviso.

In [ ]:
# Null count per column
nulos = cont.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in cont.columns
]).collect()[0].asDict()

total = cont.count()
com_nulo = {k: v for k, v in nulos.items() if v > 0}
for coluna, qtd in sorted(nulos.items()):
    print(f"  {coluna:<26} {qtd:>10,}  ({qtd/total*100:.4f}%)")

check(FONTE, "Completude", "nulos por coluna",
      STATUS_OK if not com_nulo else STATUS_ATENCAO,
      f"Colunas com nulo: {com_nulo or 'nenhuma'}.",
      "" if not com_nulo else "Avaliar impacto por coluna antes da Silver.")

In [ ]:
# Presence of each parcel across the full conjunto x ano x mes grid.
# A missing parcel does not produce a null: the row simply does not exist.
grade = cont.select("NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice", "NumPeriodoIndice").distinct()
grade_por_ano = grade.groupBy("AnoIndice").count().withColumnRenamed("count", "grao_total")

presenca = (cont
    .withColumn("SigIndicador", F.trim("SigIndicador"))
    .groupBy("AnoIndice", "SigIndicador")
    .agg(F.count("*").alias("presentes"))
    .join(grade_por_ano, "AnoIndice")
    .withColumn("ausentes", F.col("grao_total") - F.col("presentes"))
    .withColumn("cobertura_pct", F.round(F.col("presentes") / F.col("grao_total") * 100, 2))
    .orderBy("SigIndicador", "AnoIndice"))

display(presenca.filter(F.col("AnoIndice").isin(ANOS_JANELA)))

In [ ]:
# Parcels used by the composition must be complete inside the analysis window
usadas = [f"{ind}{p}" for ind in ("DEC", "FEC")
          for p in set(COMPOSICAO["desde_2022"] + PARCELAS_FI)]

faltas = (presenca
    .filter(F.col("AnoIndice").isin(ANOS_JANELA))
    .filter(F.col("SigIndicador").isin(usadas))
    .filter(F.col("ausentes") > 0)
    .orderBy(F.col("ausentes").desc()))

n_faltas = faltas.count()
if n_faltas:
    display(faltas)

check(FONTE, "Completude", "presenca das parcelas usadas",
      STATUS_OK if n_faltas == 0 else STATUS_PROBLEMA,
      f"Parcelas avaliadas: {sorted(usadas)}. "
      f"Combinacoes conjunto-ano-mes sem a parcela na janela: {n_faltas}.",
      "" if n_faltas == 0 else "Tratar ausencia como zero apenas se confirmado que a parcela nao se aplica.")

In [ ]:
# NumCon is the weight of the weighted average; it must exist everywhere
peso = (cont
    .filter(F.trim("SigIndicador") == "NumCon")
    .select("NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice", "NumPeriodoIndice", "VlrIndiceEnviado"))

peso_grao = peso.count()
peso_zero = peso.filter(F.col("VlrIndiceEnviado") <= 0).count()
peso_nulo = peso.filter(F.col("VlrIndiceEnviado").isNull()).count()
grao_total = grade.count()

print(f"grao total.................: {grao_total:,}")
print(f"grao com NumCon............: {peso_grao:,}")
print(f"NumCon nulo................: {peso_nulo:,}")
print(f"NumCon zero ou negativo....: {peso_zero:,}")

falho = (grao_total - peso_grao) + peso_zero + peso_nulo
check(FONTE, "Completude", "NumCon como peso",
      STATUS_OK if falho == 0 else STATUS_PROBLEMA,
      f"NumCon presente em {peso_grao:,} de {grao_total:,} combinacoes, "
      f"com {peso_zero:,} zeros e {peso_nulo:,} nulos.",
      "NumCon da propria base e o peso da media ponderada; dispensa fonte externa."
      if falho == 0 else "Conjuntos sem peso valido nao podem ser agregados; definir tratamento.")

### 4.3 ConsistênciaQuatro verificações, da mais simples à que mais afeta o resultado.A primeira compara os tipos publicados com o dicionário de dados da ANEEL. O dicionárioespecifica `NumCNPJ` como cadeia de 14 caracteres, `IdeConjUndConsumidoras` como cadeiade 5 e `AnoIndice` como cadeia de 4. O arquivo entrega os três como inteiro, o quedestrói zeros à esquerda nos dois primeiros e quebra qualquer junção por esses campos.A segunda testa a estabilidade dos rótulos. A ANEEL associa nomes distintos ao mesmocódigo de conjunto ao longo da série, e o mesmo ocorre com a sigla da distribuidora.Se isso se confirmar, nem `DscConjUndConsumidoras` nem `SigAgente` podem integrar achave ou a dimensão — a identidade é o código, e o nome vem do de-para próprio.A terceira verifica o domínio de `NumPeriodoIndice`.A quarta é a mais consequente: detectar reestruturação do conjunto de unidadesconsumidoras. Distribuidoras reorganizam seus conjuntos, tipicamente em janeiro. Quandoisso ocorre, conjuntos novos entram sem histórico e o acumulado de doze meses ficaartificialmente baixo. Comparar a quantidade de conjuntos em dezembro com a de janeiroseguinte, por distribuidora, revela onde houve reestruturação.

In [ ]:
# Published types against the ANEEL data dictionary
esperado = {
    "NumCNPJ": ("string", 14),
    "IdeConjUndConsumidoras": ("string", 5),
    "AnoIndice": ("string", 4),
    "SigIndicador": ("string", 3),
    "SigAgente": ("string", 20),
    "DscConjUndConsumidoras": ("string", 255),
}
publicado = dict(cont.dtypes)

divergencias = []
for campo, (tipo_dic, tamanho) in esperado.items():
    tipo_pub = publicado.get(campo, "ausente")
    if not tipo_pub.startswith(tipo_dic):
        divergencias.append(f"{campo}: dicionario {tipo_dic}({tamanho}), publicado {tipo_pub}")

for d in divergencias:
    print(f"  {d}")

check(FONTE, "Consistencia", "tipos contra o dicionario",
      STATUS_OK if not divergencias else STATUS_PROBLEMA,
      f"{len(divergencias)} campos divergem do dicionario: "
      + "; ".join(divergencias) if divergencias else "Todos os campos conferem.",
      "Normalizar na Silver: CNPJ para 14 e conjunto para 5 caracteres, com zeros a esquerda."
      if divergencias else "")

In [ ]:
# How many identifiers actually lose leading zeros when read as integer
ident = (cont
    .select(
        F.length(F.col("NumCNPJ").cast("string")).alias("len_cnpj"),
        F.length(F.col("IdeConjUndConsumidoras").cast("string")).alias("len_conj"))
    .groupBy("len_cnpj", "len_conj").count().orderBy("len_cnpj", "len_conj"))
display(ident)

cnpj_curto = cont.filter(F.length(F.col("NumCNPJ").cast("string")) < 14).count()
conj_curto = cont.filter(F.length(F.col("IdeConjUndConsumidoras").cast("string")) < 5).count()

check(FONTE, "Consistencia", "zeros a esquerda perdidos",
      STATUS_OK if (cnpj_curto + conj_curto) == 0 else STATUS_PROBLEMA,
      f"Linhas com CNPJ abaixo de 14 digitos: {cnpj_curto:,}. "
      f"Linhas com codigo de conjunto abaixo de 5 digitos: {conj_curto:,}.",
      "Aplicar lpad na Silver antes de qualquer juncao por esses campos."
      if (cnpj_curto + conj_curto) else "")

In [ ]:
# Label stability: one code should map to one name
nomes_conjunto = (cont
    .groupBy("IdeConjUndConsumidoras")
    .agg(F.countDistinct(F.trim("DscConjUndConsumidoras")).alias("nomes"))
    .filter(F.col("nomes") > 1))

siglas_agente = (cont
    .groupBy("NumCNPJ")
    .agg(F.countDistinct(F.trim("SigAgente")).alias("siglas"))
    .filter(F.col("siglas") > 1))

n_conj = nomes_conjunto.count()
n_agt = siglas_agente.count()
total_conj = cont.select("IdeConjUndConsumidoras").distinct().count()
total_agt = cont.select("NumCNPJ").distinct().count()

print(f"conjuntos com mais de um nome....: {n_conj:,} de {total_conj:,}")
print(f"CNPJs com mais de uma sigla......: {n_agt:,} de {total_agt:,}")

if n_conj:
    display(cont.join(nomes_conjunto, "IdeConjUndConsumidoras")
                .select("IdeConjUndConsumidoras", "DscConjUndConsumidoras", "AnoIndice")
                .distinct().orderBy("IdeConjUndConsumidoras", "AnoIndice").limit(40))

check(FONTE, "Consistencia", "estabilidade de rotulos",
      STATUS_OK if (n_conj + n_agt) == 0 else STATUS_ATENCAO,
      f"{n_conj} de {total_conj} conjuntos tem mais de uma descricao; "
      f"{n_agt} de {total_agt} CNPJs tem mais de uma sigla.",
      "Identificacao por codigo. Nome e sigla vem do de-para proprio, fora da chave."
      if (n_conj + n_agt) else "")

In [ ]:
# NumPeriodoIndice domain
periodos = sorted([r["NumPeriodoIndice"] for r in
                   cont.select("NumPeriodoIndice").distinct().collect()])
fora = [p for p in periodos if p < 1 or p > 12]

check(FONTE, "Consistencia", "dominio de NumPeriodoIndice",
      STATUS_OK if not fora else STATUS_PROBLEMA,
      f"Valores encontrados: {periodos}. Fora do intervalo 1 a 12: {fora or 'nenhum'}. "
      f"Confirma granularidade mensal unica, sem coexistencia de trimestral ou anual.",
      "" if not fora else "Investigar registros fora do intervalo antes de acumular.")

In [ ]:
# Restructuring of consumer unit sets: December against the following January
dez = (cont.filter(F.col("NumPeriodoIndice") == 12)
           .groupBy("NumCNPJ", "AnoIndice")
           .agg(F.countDistinct("IdeConjUndConsumidoras").alias("conj_dez")))

jan = (cont.filter(F.col("NumPeriodoIndice") == 1)
           .groupBy("NumCNPJ", "AnoIndice")
           .agg(F.countDistinct("IdeConjUndConsumidoras").alias("conj_jan"))
           .withColumn("AnoIndice", F.col("AnoIndice") - 1))

virada = (dez.join(jan, ["NumCNPJ", "AnoIndice"], "inner")
    .withColumn("delta", F.col("conj_jan") - F.col("conj_dez"))
    .withColumn("virada", F.concat_ws("/", F.lit("dez"), F.col("AnoIndice"),
                                      F.lit("jan"), F.col("AnoIndice") + 1)))

reestruturou = virada.filter(F.col("delta") != 0)
n_reest = reestruturou.count()

siglas = (cont.select("NumCNPJ", F.trim("SigAgente").alias("SigAgente")).distinct()
              .groupBy("NumCNPJ").agg(F.max("SigAgente").alias("SigAgente")))

if n_reest:
    display(reestruturou.join(siglas, "NumCNPJ")
            .select("SigAgente", "NumCNPJ", "virada", "conj_dez", "conj_jan", "delta")
            .orderBy(F.abs(F.col("delta")).desc()))

na_janela = reestruturou.filter(F.col("AnoIndice").isin(ANOS_JANELA)).count()

check(FONTE, "Consistencia", "reestruturacao de conjuntos",
      STATUS_OK if na_janela == 0 else STATUS_ATENCAO,
      f"{n_reest} viradas de ano com mudanca na quantidade de conjuntos em toda a serie; "
      f"{na_janela} dentro da janela {ANO_INICIAL}-{ANO_FINAL}.",
      "Acumular sempre em dezembro neutraliza o efeito dentro do ano civil. "
      "Distribuidoras que reestruturaram exigem cautela na comparacao entre blocos.")

### 4.4 UnicidadeO grão declarado é conjunto, ano, mês e indicador. Duplicata nesse grão significariaduplo envio da distribuidora ou falha de consolidação da ANEEL, e inflaria o indicadorao somar o mesmo valor duas vezes.

In [ ]:
chave = ["NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice", "NumPeriodoIndice", "SigIndicador"]

dup = (cont.withColumn("SigIndicador", F.trim("SigIndicador"))
           .groupBy(*chave).agg(F.count("*").alias("ocorrencias"),
                                F.countDistinct("VlrIndiceEnviado").alias("valores_distintos"))
           .filter(F.col("ocorrencias") > 1))

n_dup = dup.count()
if n_dup:
    display(dup.orderBy(F.col("ocorrencias").desc()).limit(50))
    iguais = dup.filter(F.col("valores_distintos") == 1).count()
    detalhe = (f"{n_dup:,} chaves duplicadas, das quais {iguais:,} com valor identico "
               f"e {n_dup - iguais:,} com valores divergentes.")
    trat = ("Duplicata com valor identico: deduplicar. "
            "Com valor divergente: investigar antes de escolher o registro valido.")
else:
    detalhe = "Nenhuma duplicata no grao conjunto x ano x mes x indicador."
    trat = ""

check(FONTE, "Unicidade", "duplicatas no grao declarado",
      STATUS_OK if n_dup == 0 else STATUS_PROBLEMA, detalhe, trat)

### 4.5 AcuráciaA verificação central desta fonte. O PRODIST, Módulo 8, Seção 8.2, define como o DEC eo FEC se compõem a partir das parcelas, e a definição mudou no meio da série:```até dez/2021:   DEC = DECIND + DECIP + DECXN + DECXPdesde jan/2022: DEC = DECIND + DECIP```O FEC segue as parcelas equivalentes. A verificação recompõe o indicador a partir dasparcelas e compara com o valor consolidado publicado, em cada regime. Ela prova duascoisas ao mesmo tempo: que o pipeline implementa a regra oficial corretamente, e que odado publicado adere à própria norma.Divergências aqui não são erro do pipeline. São erro da fonte, e é por isso que osindicadores deste trabalho são compostos a partir das parcelas, nunca lidos doconsolidado.

In [ ]:
# Wide layout: one row per conjunto x ano x mes, one column per indicator
todos_ind = [r["SigIndicador"] for r in
             cont.select(F.trim("SigIndicador").alias("SigIndicador")).distinct().collect()]

largo = (cont
    .withColumn("SigIndicador", F.trim("SigIndicador"))
    .groupBy("NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice", "NumPeriodoIndice")
    .pivot("SigIndicador", todos_ind)
    .agg(F.first("VlrIndiceEnviado"))
    .na.fill(0.0))

largo.cache()
print(f"grao conjunto x ano x mes: {largo.count():,} linhas, {len(largo.columns)} colunas")

In [ ]:
def testar_composicao(df, indicador, parcelas, rotulo):
    """Compare the indicator recomposed from parcels against the published aggregate."""
    colunas = [f"{indicador}{p}" for p in parcelas]
    soma = sum(F.col(c) for c in colunas)
    res = (df
        .withColumn("soma_parcelas", soma)
        .withColumn("diferenca", F.col("soma_parcelas") - F.col(indicador))
        .withColumn("adere", F.abs(F.col("diferenca")) <= TOLERANCIA))

    agg = res.groupBy("AnoIndice").agg(
        F.count("*").alias("linhas"),
        F.sum(F.col("adere").cast("int")).alias("aderentes"),
        F.max(F.abs(F.col("diferenca"))).alias("maior_desvio"))
    agg = (agg.withColumn("aderencia_pct", F.round(F.col("aderentes") / F.col("linhas") * 100, 2))
              .withColumn("regime", F.lit(rotulo))
              .withColumn("indicador", F.lit(indicador))
              .orderBy("AnoIndice"))
    return res, agg


resultados = []
for indicador in ("DEC", "FEC"):
    for rotulo, parcelas in COMPOSICAO.items():
        alvo = (F.col("AnoIndice") < ANO_MUDANCA_REGIME if rotulo == "ate_2021"
                else F.col("AnoIndice") >= ANO_MUDANCA_REGIME)
        _, agg = testar_composicao(largo.filter(alvo), indicador, parcelas, rotulo)
        resultados.append(agg)

aderencia = resultados[0]
for r in resultados[1:]:
    aderencia = aderencia.unionByName(r)

display(aderencia.select("indicador", "regime", "AnoIndice", "linhas",
                         "aderentes", "aderencia_pct", "maior_desvio")
                 .orderBy("indicador", "AnoIndice"))

In [ ]:
# Detail of the rows that do not adhere, under the regime in force since 2022
divergentes = None
for indicador in ("DEC", "FEC"):
    res, _ = testar_composicao(
        largo.filter(F.col("AnoIndice") >= ANO_MUDANCA_REGIME),
        indicador, COMPOSICAO["desde_2022"], "desde_2022")
    d = (res.filter(~F.col("adere"))
            .withColumn("indicador", F.lit(indicador))
            .select("indicador", "NumCNPJ", "IdeConjUndConsumidoras",
                    "AnoIndice", "NumPeriodoIndice",
                    F.round("soma_parcelas", 4).alias("soma_parcelas"),
                    F.round(F.col(indicador), 4).alias("publicado"),
                    F.round("diferenca", 4).alias("diferenca")))
    divergentes = d if divergentes is None else divergentes.unionByName(d)

divergentes.cache()
n_div = divergentes.count()

por_empresa = (divergentes.join(siglas, "NumCNPJ")
    .groupBy("SigAgente", "NumCNPJ", "AnoIndice", "NumPeriodoIndice", "indicador")
    .agg(F.count("*").alias("conjuntos"),
         F.round(F.min("diferenca"), 4).alias("dif_min"),
         F.round(F.max("diferenca"), 4).alias("dif_max"))
    .orderBy(F.col("conjuntos").desc()))

display(por_empresa)

check(FONTE, "Acuracia", "composicao normativa do DEC e do FEC",
      STATUS_OK if n_div == 0 else STATUS_PROBLEMA,
      f"Regime desde 2022 (IND + IP): {n_div:,} combinacoes conjunto-ano-mes em que a soma "
      f"das parcelas difere do consolidado publicado em mais de {TOLERANCIA}.",
      "Compor o indicador sempre a partir das parcelas. O consolidado da ANEEL serve "
      "apenas como conferencia, com tolerancia de arredondamento.")

In [ ]:
# Does the divergence reach the universe of interest? Concentration matters more
# than volume: an isolated error is noise, a systematic one is a finding.
concentracao = (divergentes.join(siglas, "NumCNPJ")
    .groupBy("SigAgente")
    .agg(F.count("*").alias("linhas"),
         F.countDistinct("IdeConjUndConsumidoras").alias("conjuntos"),
         F.countDistinct(F.concat_ws("-", "AnoIndice", "NumPeriodoIndice")).alias("meses"),
         F.round(F.max(F.abs(F.col("diferenca"))), 4).alias("maior_desvio"))
    .orderBy(F.col("linhas").desc()))

display(concentracao)

### 4.6 OutliersTrês frentes. Valores impossíveis, como DEC ou FEC negativo e peso não positivo, queindicam erro de envio. Valores extremos, que existem de fato — um temporal severoproduz DEC alto e legítimo —, e por isso não são removidos, apenas dimensionados, paraque se saiba quanto do resultado depende de poucos eventos. E meses faltantes dentro deum ano, que distorceriam o acumulado de doze meses sem gerar nulo algum.

In [ ]:
# Impossible values
negativos = {}
for coluna in [f"{i}{p}" for i in ("DEC", "FEC") for p in PARCELAS_FI] + ["DEC", "FEC", "NumCon"]:
    if coluna in largo.columns:
        n = largo.filter(F.col(coluna) < 0).count()
        if n:
            negativos[coluna] = n

peso_invalido = largo.filter(F.col("NumCon") <= 0).count()

check(FONTE, "Outliers", "valores impossiveis",
      STATUS_OK if not negativos and peso_invalido == 0 else STATUS_PROBLEMA,
      f"Colunas com valor negativo: {negativos or 'nenhuma'}. "
      f"Linhas com NumCon menor ou igual a zero: {peso_invalido:,}.",
      "" if not negativos and peso_invalido == 0
      else "Registros com valor impossivel nao podem compor o indicador.")

In [ ]:
# Distribution of the own indicator, DEC-FI, inside the analysis window
janela = largo.filter(F.col("AnoIndice").isin(ANOS_JANELA))

for indicador in ("DEC", "FEC"):
    colunas = [f"{indicador}{p}" for p in PARCELAS_FI]
    janela = janela.withColumn(f"{indicador}_FI", sum(F.col(c) for c in colunas))

dist = janela.select(
    F.round(F.mean("DEC_FI"), 4).alias("dec_fi_media"),
    F.round(F.expr("percentile_approx(DEC_FI, 0.5)"), 4).alias("dec_fi_mediana"),
    F.round(F.expr("percentile_approx(DEC_FI, 0.99)"), 4).alias("dec_fi_p99"),
    F.round(F.max("DEC_FI"), 4).alias("dec_fi_max"),
    F.round(F.mean("FEC_FI"), 4).alias("fec_fi_media"),
    F.round(F.expr("percentile_approx(FEC_FI, 0.5)"), 4).alias("fec_fi_mediana"),
    F.round(F.expr("percentile_approx(FEC_FI, 0.99)"), 4).alias("fec_fi_p99"),
    F.round(F.max("FEC_FI"), 4).alias("fec_fi_max"))
display(dist)

p99 = janela.selectExpr("percentile_approx(DEC_FI, 0.99) as p").collect()[0]["p"]
extremos = janela.filter(F.col("DEC_FI") > p99)
n_ext = extremos.count()
peso_ext = (extremos.agg(F.sum(F.col("DEC_FI") * F.col("NumCon"))).collect()[0][0] or 0)
peso_tot = (janela.agg(F.sum(F.col("DEC_FI") * F.col("NumCon"))).collect()[0][0] or 1)

check(FONTE, "Outliers", "concentracao nos extremos",
      STATUS_ATENCAO if peso_ext / peso_tot > 0.25 else STATUS_OK,
      f"O 1% de combinacoes com maior DEC-FI (acima de {p99:.2f} h) responde por "
      f"{peso_ext/peso_tot*100:.1f}% da duracao total ponderada da janela.",
      "Extremos sao mantidos: temporal severo produz DEC alto e legitimo. "
      "A concentracao e reportada para dimensionar a sensibilidade do ranking.")

In [ ]:
# Missing months inside a year distort the twelve month accumulation
meses_por_conjunto = (largo
    .filter(F.col("AnoIndice").isin(ANOS_JANELA))
    .groupBy("NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice")
    .agg(F.countDistinct("NumPeriodoIndice").alias("meses")))

incompletos = meses_por_conjunto.filter(F.col("meses") < 12)
n_inc = incompletos.count()
n_tot = meses_por_conjunto.count()

if n_inc:
    display(incompletos.join(siglas, "NumCNPJ")
        .groupBy("SigAgente", "AnoIndice")
        .agg(F.count("*").alias("conjuntos_incompletos"),
             F.min("meses").alias("menor_cobertura"))
        .orderBy(F.col("conjuntos_incompletos").desc()))

check(FONTE, "Outliers", "meses faltantes no ano civil",
      STATUS_OK if n_inc == 0 else STATUS_ATENCAO,
      f"{n_inc:,} de {n_tot:,} combinacoes conjunto-ano tem menos de 12 meses na janela.",
      "" if n_inc == 0 else
      "Conjunto com ano incompleto tem acumulado subestimado. Definir na Silver se e "
      "excluido do bloco ou anualizado.")

### 4.7 Síntese: problemas encontrados e tratamentos definidosConsolidação dos achados desta fonte e das decisões que eles produzem para a Silver.

In [ ]:
display(resumo_achados(FONTE))

As decisões que a Silver de continuidade herda desta seção:| Achado | Tratamento ||---|---|| Identificadores publicados como inteiro | `NumCNPJ` para texto de 14 e `IdeConjUndConsumidoras` para texto de 5, com zeros à esquerda, antes de qualquer junção || Rótulos instáveis para o mesmo código | Identificação por código; nome e sigla vêm do de-para próprio e ficam fora da chave || Consolidado da ANEEL divergente da norma | Indicador sempre composto a partir das parcelas; consolidado apenas como conferência || Composição com dois regimes | Regra de composição condicionada ao ano; na janela 2023-2025 vale apenas o regime vigente desde 2022 || Reestruturação de conjuntos em janeiro | Acumulação de doze meses sempre fechando em dezembro, o que confina o efeito a uma virada de bloco || `NumCon` completo e positivo | Peso da média ponderada, sem necessidade de fonte externa |

## Pendências documentadasFontes ainda não verificadas em profundidade, com o motivo e o que falta fazer. Aseção cresce à medida que as seções por fonte são escritas, e o que permanecer aqui naentrega é discutido na autoavaliação.| Fonte | Situação | O que falta ||---|---|---|| `emergency_occurrences_v1` e `_v2` | Não verificada | Validar o de-para entre os dois leiautes, em especial a hipótese de `DscOcorrenciaAberta` ter virado a quádrupla de fato gerador || `complaints` | Não verificada | Conferir o esquema entre os quatro anos e o filtro de nível 1 || `indger_commercial_services` e `commercial_quality` | Não verificada | Comparar universos e decidir a fonte da métrica 2 || `indger_commercial` | Não verificada | Validar `QtdUCAtiva` como denominador || `voltage_conformity` | Não verificada | Confirmar se `VlrLimite` é medida ou limite, e o tamanho da amostra || `pdd_investment` | Achados preliminares registrados | Duplicatas, CNPJ com duas siglas e valores como texto com vírgula decimal |

## Autoavaliação desta etapaA ser escrita ao final da execução, cobrindo o que a etapa entregou, o que mudou nocaminho, o que ficou em aberto e o que eu faria diferente.